# Lab A — Sensor Data Analysis for Sports
**Course:** CM2023/HI1033 Mobile Sports Applications (HT26)

**Student:** TODO: full name / last name

**Status:** Starter notebook. Complete the recording and replace every TODO before submitting.

This notebook is the lab report. The original file sensor_raw.csv must be collected from a real phone sensor recording, not simulated or edited.

## Step 1 — Data collection notes (fill in after recording)

- **Sport/activity:** TODO
- **Session location and description:** TODO (avoid publishing precise private locations on a public GitHub repository)
- **Date and duration:** TODO
- **Phone and sensor app:** TODO
- **Phone position/body part:** TODO
- **App sampling rate (target: 50 Hz):** TODO
- **Notable events with approximate timestamps:** TODO (e.g., rest/sprint/start/end)
- **Acceleration units in CSV (m/s² or g):** TODO

Recording protocol: use one constant phone position, record at least 5 minutes, start recording before moving, stop after finishing, export CSV, and save unchanged as sensor_raw.csv.


## Step 2 — Load, inspect and clean data

Install dependencies with python -m pip install -r requirements.txt. Save sensor_raw.csv beside this notebook before running.

This starter notebook supports several commonly used accelerometer column names. If your file uses different headings, fill in COLUMN_OVERRIDES below. Confirm the units shown by your app: for values in gravitational units (g), set ACCEL_UNITS to "g"; otherwise use "m/s2". If timestamps are numeric milliseconds, change TIME_UNIT to "ms".

We will remove missing/invalid essential rows, reject non-increasing timestamps and trim 1 second from each end to reduce recording start/stop artefacts. Review these choices against your real data and describe any adjustments.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAW_FILE = Path("sensor_raw.csv")
ACCEL_UNITS = "m/s2"     # "m/s2" or "g"; check your sensor app!
TIME_UNIT = "s"          # For numeric time: "s", "ms", "us", or "ns"
TRIM_EDGE_SECONDS = 1.0  # Set to 0.0 if your first/last second contains meaningful exercise
CLUSTER_GAP_SECONDS = 2.0

# Override a name only if automatic detection fails.
COLUMN_OVERRIDES = {"time": None, "ax": None, "ay": None, "az": None}

if not RAW_FILE.is_file():
    raise FileNotFoundError(
        "Record your real exercise first, export the ORIGINAL CSV, "
        "and place it beside this notebook as sensor_raw.csv."
    )
raw = pd.read_csv(RAW_FILE)
print("Shape (rows, columns):", raw.shape)
print("Column types:")
print(raw.dtypes.to_string())
print("Column names:", list(raw.columns))


In [ ]:
def normalized_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

ALIASES = {
    "time": ["time", "times", "timestamp", "loggingtime", "elapsedtime", "secondselapsed", "seconds", "datetime"],
    "ax": ["ax", "axms2", "axms", "xms2", "xms", "accelerationx", "accelerometerx", "accelerometeraccelerationx", "linearaccelerationx", "useraccelerationx", "x"],
    "ay": ["ay", "ayms2", "ayms", "yms2", "yms", "accelerationy", "accelerometery", "accelerometeraccelerationy", "linearaccelerationy", "useraccelerationy", "y"],
    "az": ["az", "azms2", "azms", "zms2", "zms", "accelerationz", "accelerometerz", "accelerometeraccelerationz", "linearaccelerationz", "useraccelerationz", "z"],
}

def find_column(kind):
    if COLUMN_OVERRIDES[kind] is not None:
        selected = COLUMN_OVERRIDES[kind]
        if selected not in raw.columns:
            raise ValueError(f"Column override {selected!r} not found.")
        return selected
    candidates = [(col, normalized_name(col)) for col in raw.columns]
    # Prefer exact matches, then long aliases followed by unit suffixes.
    for alias in ALIASES[kind]:
        for col, normalized in candidates:
            if normalized == alias:
                return col
    for alias in ALIASES[kind]:
        if len(alias) >= 4:
            for col, normalized in candidates:
                if normalized.startswith(alias):
                    return col
    raise ValueError(
        f"No {kind} column found. Available: {list(raw.columns)}. "
        "Set COLUMN_OVERRIDES in the previous cell and rerun."
    )

cols = {kind: find_column(kind) for kind in ALIASES}
if len(set(cols.values())) != 4:
    raise ValueError(f"Column selection has duplicates: {cols}")
print("Selected columns:", cols)

series_time = raw[cols["time"]]
numeric_time = pd.to_numeric(series_time, errors="coerce")
if numeric_time.notna().sum() >= 0.9 * len(series_time):
    unit_scale = {"s": 1.0, "ms": 1000.0, "us": 1e6, "ns": 1e9}
    if TIME_UNIT not in unit_scale:
        raise ValueError("TIME_UNIT must be s, ms, us or ns")
    seconds = numeric_time / unit_scale[TIME_UNIT]
else:
    timestamps = pd.to_datetime(series_time, errors="coerce", utc=True)
    seconds = (timestamps - timestamps.iloc[0]).dt.total_seconds()

data = pd.DataFrame({"time": seconds})
for axis in ("ax", "ay", "az"):
    data[axis] = pd.to_numeric(raw[cols[axis]], errors="coerce")

if ACCEL_UNITS == "g":
    data[["ax", "ay", "az"]] *= 9.80665
elif ACCEL_UNITS != "m/s2":
    raise ValueError('ACCEL_UNITS must be "m/s2" or "g"')

print("First 10 selected sensor readings (without optional GPS columns):")
display(data.head(10))
print("Numeric summary before cleaning:")
display(data.describe())


In [ ]:
data = data.replace([np.inf, -np.inf], np.nan)
missing_by_column = data.isna().sum()
missing_rows = int(data.isna().any(axis=1).sum())
print("Missing/invalid values:", missing_by_column.to_dict())

valid = data.dropna(subset=["time", "ax", "ay", "az"]).copy()
bad_order = valid["time"].diff().le(0)
bad_order_count = int(bad_order.sum())
valid = valid.loc[~bad_order].copy()

edge_removed = 0
if TRIM_EDGE_SECONDS > 0 and len(valid) > 2:
    start, end = valid["time"].iloc[0], valid["time"].iloc[-1]
    if end - start > 2 * TRIM_EDGE_SECONDS + 2:
        before = len(valid)
        valid = valid.loc[
            (valid["time"] >= start + TRIM_EDGE_SECONDS)
            & (valid["time"] <= end - TRIM_EDGE_SECONDS)
        ].copy()
        edge_removed = before - len(valid)

df_clean = valid.reset_index(drop=True)
if len(df_clean) < 3:
    raise ValueError("Too few sensor samples after cleaning. Inspect your CSV and settings.")
df_clean["time"] -= df_clean["time"].iloc[0]
df_clean["acc_mag"] = np.sqrt(
    df_clean["ax"] ** 2 + df_clean["ay"] ** 2 + df_clean["az"] ** 2
)
saturation_samples = int(df_clean[["ax", "ay", "az"]].abs().gt(19.6).any(axis=1).sum())
print(f"Original samples: {len(raw):,}")
print(f"Dropped missing/invalid essential rows: {missing_rows:,}")
print(f"Dropped non-increasing timestamps: {bad_order_count:,}")
print(f"Trimmed edge samples: {edge_removed:,}")
print(f"Remaining samples: {len(df_clean):,}")
print(f"Samples exceeding ±19.6 m/s² on an axis (2g sensor reference): {saturation_samples:,}")
print("High values are flagged for inspection, not removed automatically; actual sensor range may differ.")
df_clean.to_csv("sensor_clean.csv", index=False)
print("Saved cleaned data to sensor_clean.csv (do not replace sensor_raw.csv).")


### Cleaning decisions (edit this after running)

TODO: Report the actual number of missing rows, timestamp issues, and edge samples removed above. Explain **why** the cleaning was appropriate for your activity. The default 1-second trim is intended to reduce handling noise, but should be changed if those seconds represent meaningful movement. Discuss any saturation warnings and your sensor's actual range. Do not treat every movement spike as noise.


## Step 3 — Descriptive statistics and IQR anomalies

Use the median time difference to estimate the **effective** sampling rate, rather than assuming the configured 50 Hz. Discuss whether the resulting Nyquist frequency (half the sampling rate) is adequate for your activity.


In [ ]:
deltas = df_clean["time"].diff().dropna()
dt = float(deltas.median())
if not np.isfinite(dt) or dt <= 0:
    raise ValueError("Invalid median sample interval; check timestamps.")
fs = 1.0 / dt
duration_s = float(df_clean["time"].iloc[-1] - df_clean["time"].iloc[0])
stats_table = pd.DataFrame(
    [
        ("Mean acceleration magnitude", df_clean["acc_mag"].mean(), "m/s²"),
        ("Std deviation", df_clean["acc_mag"].std(), "m/s²"),
        ("Maximum", df_clean["acc_mag"].max(), "m/s²"),
        ("Minimum", df_clean["acc_mag"].min(), "m/s²"),
        ("Duration after cleaning", duration_s, "s"),
        ("Sample count", len(df_clean), "samples"),
        ("Effective sampling rate", fs, "Hz"),
        ("Nyquist frequency", fs / 2, "Hz"),
    ],
    columns=["Metric", "Value", "Unit"],
)
display(stats_table.round({"Value": 2}))
print("50 Hz target reached?", "approximately yes" if 45 <= fs <= 55 else "not exactly; discuss in notebook")


**Sampling-rate interpretation (fill in):** TODO: Compare your effective Hz with the app's setting. Explain whether that was sufficient for your movement. For context, walking/running gait cycles are often about 1–3 Hz; high-impact movements can contain higher frequencies. Do not claim to have resolved faster movements than your actual sampling rate supports.


In [ ]:
q1 = df_clean["acc_mag"].quantile(0.25)
q3 = df_clean["acc_mag"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df_clean.loc[
    (df_clean["acc_mag"] < lower) | (df_clean["acc_mag"] > upper)
].copy()
percent = 100.0 * len(outliers) / len(df_clean)
print(f"IQR thresholds: {lower:.3f} to {upper:.3f} m/s²")
print(f"Outliers: {len(outliers):,} ({percent:.2f}% of samples)")
display(outliers[["time", "acc_mag"]].head(10))

if not outliers.empty:
    outliers["cluster"] = (
        outliers["time"].diff().fillna(np.inf).gt(CLUSTER_GAP_SECONDS).cumsum()
    )
    clusters = outliers.groupby("cluster", as_index=False).agg(
        start_s=("time", "min"),
        end_s=("time", "max"),
        samples=("time", "size"),
        peak_acc_m_s2=("acc_mag", "max"),
    )
    display(clusters.round(2))
    print(f"Found {len(clusters)} anomaly clusters using a {CLUSTER_GAP_SECONDS}s separation.")
else:
    clusters = pd.DataFrame(columns=["cluster", "start_s", "end_s", "samples", "peak_acc_m_s2"])
    print("No IQR outliers; explain why your activity may produce a narrow distribution.")


### Physical explanation of anomaly clusters (required)

TODO: For **every cluster** listed above, record the approximate time interval, the likely physical event (compare with your session notes), and whether it is real sport movement or recording error. The IQR method detects unusual numbers, **not** automatically faulty readings. If many clusters appear, consider increasing CLUSTER_GAP_SECONDS to group nearby events, and explain the grouping.

- Cluster/time interval: TODO
- Likely physical cause and evidence: TODO
- Keep as meaningful signal or exclude as artefact? Why? TODO


## Step 4 — Three matplotlib plots

All plots below are generated in Python with titles and axis labels. After running each plot, write **2–4 sentences** that say what it shows, what is interesting/surprising, and what it tells you about the sport or your performance.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_clean["time"], df_clean["acc_mag"], linewidth=0.55, label="Acceleration magnitude")
ax.axhline(df_clean["acc_mag"].mean(), linestyle="--", label="Mean")
if not outliers.empty:
    ax.scatter(outliers["time"], outliers["acc_mag"], s=8, label="IQR outlier", alpha=0.7)
ax.set_xlabel("Time since cleaned recording start (s)")
ax.set_ylabel("Acceleration magnitude (m/s²)")
ax.set_title("Plot 1 — Acceleration magnitude over the session")
ax.legend()
plt.tight_layout()
plt.savefig("fig1_timeseries.png", dpi=150)
plt.show()


**Plot 1 interpretation (2–4 sentences):** TODO: Explain changes over time and connect marked anomalies to documented activity events.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df_clean["acc_mag"], bins=60, edgecolor="white")
axes[0].set_xlabel("Acceleration magnitude (m/s²)")
axes[0].set_ylabel("Sample count")
axes[0].set_title("Acceleration distribution")
axes[1].boxplot(df_clean["acc_mag"].dropna(), tick_labels=["Magnitude"])
axes[1].set_ylabel("Acceleration magnitude (m/s²)")
axes[1].set_title("Acceleration box plot")
fig.suptitle("Plot 2 — Statistical distribution")
plt.tight_layout()
plt.savefig("fig2_distribution.png", dpi=150)
plt.show()


**Plot 2 interpretation (2–4 sentences):** TODO: Discuss distribution shape, spread, median, possible skew, and unusual values. Compare warm-up or cooldown if your session contains clearly marked phases.


In [ ]:
# Rolling RMS of the deviation from the session median magnitude.
# This is a movement-variation proxy, NOT a calibrated energy or effort measurement.
window_samples = max(2, int(round(2 * fs)))
deviation = df_clean["acc_mag"] - df_clean["acc_mag"].median()
rolling_rms = np.sqrt(
    deviation.pow(2).rolling(window=window_samples, min_periods=max(2, window_samples // 2)).mean()
)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_clean["time"], rolling_rms, linewidth=0.8)
ax.set_xlabel("Time since cleaned recording start (s)")
ax.set_ylabel("2-second rolling RMS of deviation (m/s²)")
ax.set_title("Plot 3 — Variation in acceleration over the session")
plt.tight_layout()
plt.savefig("fig3_rolling_rms.png", dpi=150)
plt.show()


**Plot 3 interpretation (2–4 sentences):** TODO: Explain where movement variation increased or decreased and connect it to your session notes. Higher rolling RMS means greater variation in acceleration, not a direct measurement of calories or physical effort.


## Step 5 — Market and research context

### 5.1 Peer-reviewed paper (same sport and sensor type)

**APA reference:** TODO: Real authors, year, title, journal/conference, volume/pages and DOI/URL.

1. **What did the study measure, and how does it compare with your computed feature?** TODO, use real numbers from the notebook.
2. **What sampling rate did the paper use versus your effective rate? Was your rate sufficient for the same purpose?** TODO; cite the actual study's rate.
3. **What did the study conclude? Does your data support or contradict it (informally)?** TODO; discuss limitations and avoid unsupported causal conclusions.

### 5.2 Commercial device/app (same activity)

**Product and official source link:** TODO (e.g. Garmin, Apple Watch, Whoop, Polar).

- **Its sensors:** TODO; verify using product documentation.
- **Metrics shown to the user:** TODO.
- **Metrics available to coaches/researchers by API or export:** TODO; distinguish confirmed exports from assumptions.
- **Gap between consumer output and what your peer-reviewed paper suggests matters:** TODO, based on the actual comparison.

### 5.3 Reflection (150–250 words)

TODO: Write 150–250 words **in your own words** addressing all three points:
- Where does consumer sports technology lag behind peer-reviewed research?
- Could your raw sensor measurements, processed differently, help close this gap?
- What would a sports scientist need that ordinary consumer apps fail to provide?

**Word count:** TODO, check before submission.

### Final verification

- Real, unchanged sensor_raw.csv (submit separately to Canvas)
- All notebook cells run from top to bottom with visible outputs
- Session details and actual sensor units documented
- Cleaning justified and effective sampling rate reported
- Statistics table and physically explained IQR clusters
- Three labeled plots with 2–4 interpretation sentences each
- Real peer-reviewed study in APA and three answers
- Real commercial product comparison and research–market gap
- 150–250-word reflection
- Rename this file from laba_lastname.ipynb to laba_YOURACTUALLASTNAME.ipynb before submitting
